In [1]:
import pandas as pd

In [2]:
predicted_path = r"C:\Users\L E G I O N\Documents\Programming\Python\University\NLP\EAMT\predictions_de_DE_FIXED.jsonl"
mapped_path = r"C:\Users\L E G I O N\Documents\Programming\Python\University\NLP\EAMT\ParsedData\translated_entity.csv"

## Load the Data

In [3]:
import json
import re

# Load predictions
predictions = []
with open(predicted_path, 'r', encoding='utf-8') as f:
    for line in f:
        predictions.append(json.loads(line))

print(f"Loaded {len(predictions)} predictions")
print(f"\nSample prediction:")
print(predictions[0])

# Load entity mappings
entity_mappings = pd.read_csv(mapped_path)
print(f"\nLoaded {len(entity_mappings)} entity mappings")
print(f"\nEntity mapping columns: {entity_mappings.columns.tolist()}")
print(f"\nSample mappings:")
print(entity_mappings.head())

Loaded 5876 predictions

Sample prediction:
{'id': 'bc577b19fe3bd34e', 'source_language': 'English', 'target_language': 'German', 'text': 'Who directed <entity1>?', 'prediction': 'Wer hat <<entity1>> geleitet?'}

Loaded 6555 entity mappings

Entity mapping columns: ['sentence_id', 'entity_tag', 'entity_category', 'entity_src', 'entity_translated']

Sample mappings:
        sentence_id entity_tag entity_category  \
0  bc577b19fe3bd34e  <entity1>       ['Movie']   
1  b39ba50cccda50ea  <entity1>       ['Movie']   
2  96aa8c7a91d9994e  <entity1>       ['Movie']   
3  b62e95157e1356e2  <entity1>     ['Artwork']   
4  5aa062e2ba8bc119  <entity1>     ['Artwork']   

                              entity_src  \
0  American Murder: The Family Next Door   
1  American Murder: The Family Next Door   
2  American Murder: The Family Next Door   
3                     Confederate States   
4                     Confederate States   

                       entity_translated  
0  American Murder: The

## Create Entity Mapping Dictionary

In [4]:
# Create a mapping dictionary: {sentence_id: {entity_tag: translated_value}}
entity_map = {}

for _, row in entity_mappings.iterrows():
    sentence_id = row['sentence_id']
    entity_tag = row['entity_tag']  # e.g., '<entity1>'
    translated_value = row['entity_translated']
    
    if sentence_id not in entity_map:
        entity_map[sentence_id] = {}
    
    entity_map[sentence_id][entity_tag] = translated_value

print(f"Created mappings for {len(entity_map)} sentences")
print(f"\nSample mapping for first sentence:")
sample_id = list(entity_map.keys())[0]
print(f"Sentence ID: {sample_id}")
print(f"Mappings: {entity_map[sample_id]}")

Created mappings for 5345 sentences

Sample mapping for first sentence:
Sentence ID: bc577b19fe3bd34e
Mappings: {'<entity1>': 'American Murder: The Family Next Door'}


## Remap Entities in Predictions

In [5]:
def remap_entities(text, sentence_id, entity_mappings):
    """
    Replace entity tags with their translated values and remove any remaining angle brackets.
    
    Args:
        text: The translated text with entity tags like <entity1>
        sentence_id: The ID of the sentence to look up mappings
        entity_mappings: Dictionary mapping sentence_id -> {entity_tag: translated_value}
    
    Returns:
        Text with entity tags replaced by actual translated values and all angle brackets removed
    """
    if sentence_id not in entity_mappings:
        # No mappings for this sentence, return as-is
        return text
    
    mappings = entity_mappings[sentence_id]
    
    # Find all entity tags in the text
    entity_pattern = re.compile(r'<entity\d+>')
    entities_found = entity_pattern.findall(text)
    
    # Replace each entity tag with its translated value
    result = text
    for entity_tag in entities_found:
        if entity_tag in mappings:
            translated_value = mappings[entity_tag]
            result = result.replace(entity_tag, translated_value)
    
    # Remove any remaining angle brackets
    result = result.replace('<', '').replace('>', '')
    
    return result

# Test the function
test_id = list(entity_map.keys())[0]
test_pred = [p for p in predictions if p['id'] == test_id][0] if test_id in [p['id'] for p in predictions] else None

if test_pred:
    print("Testing remapping function:")
    print("="*70)
    print(f"Original: {test_pred['prediction']}")
    remapped = remap_entities(test_pred['prediction'], test_id, entity_map)
    print(f"Remapped: {remapped}")
    print("="*70)

Testing remapping function:
Original: Wer hat <<entity1>> geleitet?
Remapped: Wer hat American Murder: The Family Next Door geleitet?


## Apply Remapping to All Predictions

In [6]:
remapped_predictions = []
skipped_count = 0
remapped_count = 0
entity_pattern = re.compile(r'<entity\d+>')

for pred in predictions:
    sentence_id = pred['id']
    original_prediction = pred['prediction']
    
    # Check if there are any entity tags in the prediction
    entities_in_prediction = entity_pattern.findall(original_prediction)
    
    if entities_in_prediction:
        # Try to remap
        remapped_prediction = remap_entities(original_prediction, sentence_id, entity_map)
        
        # Check if any entities remain (meaning they weren't found in mappings)
        remaining_entities = entity_pattern.findall(remapped_prediction)
        
        if remaining_entities:
            # Some entities couldn't be mapped - skip this prediction
            skipped_count += 1
            continue
        else:
            # Successfully remapped all entities
            remapped_count += 1
    else:
        # No entities in prediction, use as-is
        remapped_prediction = original_prediction
    
    # Add to remapped predictions
    remapped_predictions.append({
        "id": pred['id'],
        "source_language": pred['source_language'],
        "target_language": pred['target_language'],
        "text": pred['text'],
        "prediction": remapped_prediction
    })

print("="*70)
print("REMAPPING SUMMARY")
print("="*70)
print(f"Total original predictions: {len(predictions)}")
print(f"Successfully remapped: {remapped_count}")
print(f"No entities (kept as-is): {len(predictions) - remapped_count - skipped_count}")
print(f"Skipped (missing mappings): {skipped_count}")
print(f"Final prediction count: {len(remapped_predictions)}")
print("="*70)

REMAPPING SUMMARY
Total original predictions: 5876
Successfully remapped: 5117
No entities (kept as-is): 759
Skipped (missing mappings): 0
Final prediction count: 5876


## Show Sample Remapped Predictions

In [7]:
# Show examples of remapped predictions
print("SAMPLE REMAPPED PREDICTIONS")
print("="*70)

# Find predictions that were remapped (had entities)
remapped_samples = []
for pred in remapped_predictions[:100]:  # Check first 100
    # Find corresponding original prediction
    orig = [p for p in predictions if p['id'] == pred['id']][0]
    if entity_pattern.search(orig['prediction']):
        remapped_samples.append((orig, pred))
        if len(remapped_samples) >= 10:
            break

for i, (orig, remapped) in enumerate(remapped_samples[:10], 1):
    print(f"\n[Sample {i}] ID: {remapped['id']}")
    print(f"Source:   {remapped['text'][:80]}...")
    print(f"Before:   {orig['prediction'][:100]}...")
    print(f"After:    {remapped['prediction'][:100]}...")
    print("-" * 70)

print("\n" + "="*70)

SAMPLE REMAPPED PREDICTIONS

[Sample 1] ID: bc577b19fe3bd34e
Source:   Who directed <entity1>?...
Before:   Wer hat <<entity1>> geleitet?...
After:    Wer hat American Murder: The Family Next Door geleitet?...
----------------------------------------------------------------------

[Sample 2] ID: b39ba50cccda50ea
Source:   When was the movie <entity1> released?...
Before:   Wann wurde der Film <<entity1>> veröffentlicht?...
After:    Wann wurde der Film American Murder: The Family Next Door veröffentlicht?...
----------------------------------------------------------------------

[Sample 3] ID: 96aa8c7a91d9994e
Source:   Is <entity1> based on a true story?...
Before:   Ist <<entity1>> auf einer wahren Geschichte basiert?...
After:    Ist American Murder: The Family Next Door auf einer wahren Geschichte basiert?...
----------------------------------------------------------------------

[Sample 4] ID: b62e95157e1356e2
Source:   Where is the Seal of the <entity1> currently displayed?...
Be

## Save Remapped Predictions

In [8]:
output_path = r"C:\Users\L E G I O N\Documents\Programming\Python\University\NLP\EAMT\predictions_de_DE_REMAPPED.jsonl"

# Save remapped predictions
with open(output_path, 'w', encoding='utf-8') as f:
    for pred in remapped_predictions:
        json.dump(pred, f, ensure_ascii=False)
        f.write('\n')

print("="*70)
print("SAVED REMAPPED PREDICTIONS")
print("="*70)
print(f"Output file: {output_path}")
print(f"Total predictions saved: {len(remapped_predictions)}")

# Verify the file
with open(output_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()
    print(f"\nVerified {len(lines)} lines in output file")
    
    # Check first prediction
    first_pred = json.loads(lines[0])
    print(f"\n✓ Valid JSON format")
    print(f"\nFirst prediction:")
    print(f"  ID: {first_pred['id']}")
    print(f"  Source: {first_pred['text'][:60]}...")
    print(f"  Prediction: {first_pred['prediction'][:60]}...")

print("\n" + "="*70)
print("✅ REMAPPING COMPLETE")
print("="*70)
print(f"📁 Files:")
print(f"  Original: predictions_de_DE_FIXED.jsonl ({len(predictions)} predictions)")
print(f"  Remapped: predictions_de_DE_REMAPPED.jsonl ({len(remapped_predictions)} predictions)")
print(f"\n🎯 Use predictions_de_DE_REMAPPED.jsonl for final submission!")
print("="*70)

SAVED REMAPPED PREDICTIONS
Output file: C:\Users\L E G I O N\Documents\Programming\Python\University\NLP\EAMT\predictions_de_DE_REMAPPED.jsonl
Total predictions saved: 5876

Verified 5876 lines in output file

✓ Valid JSON format

First prediction:
  ID: bc577b19fe3bd34e
  Source: Who directed <entity1>?...
  Prediction: Wer hat American Murder: The Family Next Door geleitet?...

✅ REMAPPING COMPLETE
📁 Files:
  Original: predictions_de_DE_FIXED.jsonl (5876 predictions)
  Remapped: predictions_de_DE_REMAPPED.jsonl (5876 predictions)

🎯 Use predictions_de_DE_REMAPPED.jsonl for final submission!


## Format for Submission - Convert to Required Format

In [9]:
# Load the remapped predictions
input_file = r"C:\Users\L E G I O N\Documents\Programming\Python\University\NLP\EAMT\predictions_de_DE_REMAPPED.jsonl"
output_file = r"C:\Users\L E G I O N\Documents\Programming\Python\University\NLP\EAMT\de_DE.jsonl"
original_test_file = r"C:\Users\L E G I O N\Documents\Programming\Python\University\NLP\EAMT\semeval_test\test_without_targets\de_DE.jsonl"

# Read predictions
with open(input_file, 'r', encoding='utf-8') as f:
    predictions = [json.loads(line) for line in f]

# Read original test data to get the source text
with open(original_test_file, 'r', encoding='utf-8') as f:
    original_data = [json.loads(line) for line in f]

# Create a mapping from id to source text
id_to_source = {item['id']: item['source'] for item in original_data}

print(f"Loaded {len(predictions)} predictions from {input_file}")
print(f"Loaded {len(original_data)} original test entries from {original_test_file}")
print(f"\nCurrent format (sample):")
print(json.dumps(predictions[0], indent=2, ensure_ascii=False))

# Convert to required format using original source text
formatted_predictions = []
for pred in predictions:
    pred_id = pred["id"]
    
    # Get the original source text (without entity tags)
    original_source = id_to_source.get(pred_id)
    
    if original_source is None:
        print(f"⚠️ Warning: No original source found for ID {pred_id}")
        continue
    
    formatted_pred = {
        "id": pred_id,
        "source_language": "English",  # Always English for this task
        "target_language": "German",  # Can use "German" or "de" per requirements
        "text": original_source,  # Use original source text WITHOUT entity tags
        "prediction": pred["prediction"]
    }
    formatted_predictions.append(formatted_pred)

print(f"\n{'='*70}")
print("CONVERSION SUMMARY")
print(f"{'='*70}")
print(f"Total predictions: {len(formatted_predictions)}")
print(f"\nRequired format (sample):")
print(json.dumps(formatted_predictions[0], indent=2, ensure_ascii=False))
print(f"\nChanges applied:")
print(f"  • Target language: 'German' (full name format)")
print(f"  • Source language: 'English' (as required)")
print(f"  • Text: Using original source WITHOUT entity tags")
print(f"{'='*70}")

Loaded 5876 predictions from C:\Users\L E G I O N\Documents\Programming\Python\University\NLP\EAMT\predictions_de_DE_REMAPPED.jsonl
Loaded 5876 original test entries from C:\Users\L E G I O N\Documents\Programming\Python\University\NLP\EAMT\semeval_test\test_without_targets\de_DE.jsonl

Current format (sample):
{
  "id": "bc577b19fe3bd34e",
  "source_language": "English",
  "target_language": "German",
  "text": "Who directed <entity1>?",
  "prediction": "Wer hat American Murder: The Family Next Door geleitet?"
}

CONVERSION SUMMARY
Total predictions: 5876

Required format (sample):
{
  "id": "bc577b19fe3bd34e",
  "source_language": "English",
  "target_language": "German",
  "text": "Who directed American Murder: The Family Next Door?",
  "prediction": "Wer hat American Murder: The Family Next Door geleitet?"
}

Changes applied:
  • Target language: 'German' (full name format)
  • Source language: 'English' (as required)
  • Text: Using original source WITHOUT entity tags


## Save Final Submission File

In [10]:
# Save formatted predictions in submission format
with open(output_file, 'w', encoding='utf-8') as f:
    for pred in formatted_predictions:
        json.dump(pred, f, ensure_ascii=False)
        f.write('\n')

print(f"{'='*70}")
print("SUBMISSION FILE SAVED")
print(f"{'='*70}")
print(f"Output file: {output_file}")
print(f"Total predictions: {len(formatted_predictions)}")

# Verify the file
with open(output_file, 'r', encoding='utf-8') as f:
    lines = f.readlines()
    print(f"\n✓ Verified {len(lines)} lines in output file")
    
    # Check a few samples
    print(f"\n✓ Sample entries:")
    for i in [0, 1, 2]:
        sample = json.loads(lines[i])
        print(f"\n  Entry {i+1}:")
        print(f"    ID: {sample['id']}")
        print(f"    Source Lang: {sample['source_language']}")
        print(f"    Target Lang: {sample['target_language']}")
        print(f"    Text: {sample['text'][:50]}...")
        print(f"    Prediction: {sample['prediction'][:60]}...")

print(f"\n{'='*70}")
print("✅ SUBMISSION FORMAT COMPLETE")
print(f"{'='*70}")
print(f"📁 Final submission file: de_DE.jsonl")
print(f"📊 Total predictions: {len(formatted_predictions)}")
print(f"\n⚠️  NOTE: Skipped predictions will receive a score of 0")
print(f"   You have {len(formatted_predictions)} valid predictions")
print(f"\n🎯 Use de_DE.jsonl for submission to SemEval 2025 Task 2!")
print(f"{'='*70}")

SUBMISSION FILE SAVED
Output file: C:\Users\L E G I O N\Documents\Programming\Python\University\NLP\EAMT\de_DE.jsonl
Total predictions: 5876

✓ Verified 5876 lines in output file

✓ Sample entries:

  Entry 1:
    ID: bc577b19fe3bd34e
    Source Lang: English
    Target Lang: German
    Text: Who directed American Murder: The Family Next Door...
    Prediction: Wer hat American Murder: The Family Next Door geleitet?...

  Entry 2:
    ID: b39ba50cccda50ea
    Source Lang: English
    Target Lang: German
    Text: When was the movie American Murder: The Family Nex...
    Prediction: Wann wurde der Film American Murder: The Family Next Door ve...

  Entry 3:
    ID: 96aa8c7a91d9994e
    Source Lang: English
    Target Lang: German
    Text: Is American Murder: The Family Next Door based on ...
    Prediction: Ist American Murder: The Family Next Door auf einer wahren G...

✅ SUBMISSION FORMAT COMPLETE
📁 Final submission file: de_DE.jsonl
📊 Total predictions: 5876

⚠️  NOTE: Skipped predi